# Parallel Architecture — LangGraph

**Pattern:** Parallel via `Send()` API  
**Framework:** LangGraph  
**Task:** Fan-out to 3 city researchers → aggregate into ranking

## Graph with Dynamic Fan-out

```
START
  │
  ▼ (conditional edge using Send())
fan_out ──Send("research_city", {city: "Tokyo"})
        ──Send("research_city", {city: "Paris"})
        ──Send("research_city", {city: "Bangalore"})
              │          │            │
              └──────────┼────────────┘
                         ▼  (all branches join here)
                     aggregate
                         │
                        END
```

**`Send()` API:** Dispatches a node with specific sub-state.  
All `Send()` calls from `fan_out` run simultaneously as independent branches.  
Results accumulate in `city_reports` via the `operator.add` reducer.

In [ ]:
import os
from typing import TypedDict, Annotated
import operator
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in .env"
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
print("✓ Ready")

In [ ]:
CITY_DATA = {
    "tokyo":     "Weather: Clear, 18°C, score 9/10. Safety: Low, score 10/10. Time: 22:30 JST.",
    "paris":     "Weather: Partly Cloudy, 16°C, score 7/10. Safety: Low, score 8/10. Time: 15:30 CET.",
    "bangalore": "Weather: Rainy, 25°C, score 6/10. Safety: Medium, score 6/10. Time: 20:00 IST.",
}

class OverallState(TypedDict):
    cities: list
    city_reports: Annotated[list, operator.add]  # reducer: appends each branch's result
    final_ranking: str

class CityState(TypedDict):
    city: str

print("State schemas defined")
print("  OverallState.city_reports uses operator.add reducer — accumulates across parallel branches")

In [ ]:
def fan_out(state: OverallState):
    """Return Send() for each city — LangGraph dispatches them in parallel."""
    return [Send("research_city", {"city": c}) for c in state["cities"]]

def research_city(state: CityState) -> dict:
    city = state["city"]
    raw = CITY_DATA.get(city.lower(), "No data.")
    response = llm.invoke([
        SystemMessage(content="Write a structured 3-line city travel report: Weather, Safety, Recommendation."),
        HumanMessage(content=f"City: {city}\nData: {raw}"),
    ])
    print(f"  [parallel branch] {city} done")
    return {"city_reports": [f"### {city}\n{response.content}"]}

def aggregate(state: OverallState) -> dict:
    combined = "\n\n".join(state["city_reports"])
    response = llm.invoke([
        SystemMessage(content="Rank these cities best to worst by weather + safety. Justify each rank."),
        HumanMessage(content=f"Reports:\n\n{combined}\n\nRank and name the top pick."),
    ])
    return {"final_ranking": response.content}

print("3 nodes defined: fan_out (dispatcher) | research_city (parallel) | aggregate")

In [ ]:
builder = StateGraph(OverallState)
builder.add_node("research_city", research_city)
builder.add_node("aggregate", aggregate)

builder.add_conditional_edges(START, fan_out, ["research_city"])
builder.add_edge("research_city", "aggregate")
builder.add_edge("aggregate", END)

graph = builder.compile()

try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

In [ ]:
cities = ["Tokyo", "Paris", "Bangalore"]
print(f"Running parallel research for: {cities}")

result = graph.invoke({"cities": cities, "city_reports": [], "final_ranking": ""})

print("\n--- City Reports ---")
for report in result["city_reports"]:
    print(report)
    print()

print("--- Final Ranking ---")
print(result["final_ranking"])

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| `Send("node", sub_state)` | Dynamic fan-out — dispatches N copies of the same node |
| `Annotated[list, operator.add]` reducer | Accumulates results from all parallel branches into one list |
| `fan_out` returns a list of `Send()` | LangGraph executes all branches before `aggregate` runs |
| `aggregate` node waits for all branches | LangGraph's join semantics: all incoming edges must complete first |

### When to Use `Send()` vs `asyncio.gather()`
| | LangGraph `Send()` | LangChain `asyncio.gather()` |
|---|---|---|
| Visualization | Graph shows fan-out | No built-in visualization |
| State tracking | TypedDict state per branch | Variables |
| Adding retries | Add conditional edge | Wrap in try/except |
| Checkpointing | Built-in with MemorySaver | Not available |

### Next: [CrewAI Parallel →](../CrewAI/parallel.ipynb)